In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
# from catboost import CatBoostClassifier


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

full_path = os.path.join(path, 'Q3_data.csv')

df = pd.read_csv(full_path)

In [ ]:
# Task 2: Write your code here:

df.head()

In [ ]:
# Task 3: Write your code here:

df.info()

In [ ]:
# Task 4: Write your code here:

df.describe()

In [ ]:
# Task 1: Write your code here:

df_clean = df.copy()

df_clean

for col in df_clean.columns:

  if(df_clean[col].isnull().sum().sum() > 4):

    df_clean = df_clean.drop(columns = col)

  else:

    df_clean = df_clean.fillna(df_clean[col].mean())

df_clean

In [ ]:
# Task 2: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 3: Write your code here:

categorical_cols = df_clean.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    # TODO: Apply fit_transform to encode the column
    df_clean[col] = le.fit_transform(df_clean[col])

df_clean.head()

# its not needed as only datatypes are float and int, but it deosnt hurt... hopefully

In [ ]:
# Task 4: Write your code here:

features = df_clean.columns.drop("Target")

scaler = StandardScaler()
df_clean[features] = scaler.fit_transform(df_clean[features])

df_clean.head()

In [ ]:
# Task 5: Write your code here:


import seaborn as sns

# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_clean, "Target")

In [ ]:
# Task 1: Write your code here:

X = df_clean.drop("Target", axis=1).astype(float)

y = df_clean['Target'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:

%pip install kagglehub catboost lightgbm tqdm -q

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from catboost import CatBoostClassifier

n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

#

cb = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4)

average_f1 = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):

  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  print(f"Training model...")
  cb.fit(X_train, y_train) # train
  y_pred = cb.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  f1 = f1_score(y_test, y_pred, zero_division=0)

  average_f1.append(f1)

print(f"  F1 for all folds:  {np.mean(average_f1):.4f}")


In [ ]:
# Task 1: Write your code here:

from sklearn.linear_model import Ridge, Lasso

# Gather importances from the models (from the last fold)
importances = {}

importances['catboost'] = cb.feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 2, figsize=(45, 25))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

# plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

print('most important feature is: D-45, as shown by the plot.')

In [ ]:
# Task Bonus: Write your code here:

def bonus():

  X = df_clean['D_45'].astype(float)

  y = df_clean['Target'].astype(float)

  n_splits = 5 # K=5 Folds

  # Stratified 5-Fold Cross-Validation, shuffled
  skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

  #

  cb = CatBoostClassifier(
    verbose=0,
    n_estimators=320,
    max_depth=4)

  average_f1 = []

  for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):

    print(f"\nFold {fold_idx + 1}/{n_splits}")

    # 1. Split data
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 2. Train & Validate sklearn models
    print(f"Training model...")
    cb.fit(X_train, y_train) # train
    y_pred = cb.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
    f1 = f1_score(y_test, y_pred, zero_division=0)

    average_f1.append(f1)

  print(f"  F1 for all folds:  {np.mean(average_f1):.4f}")

bonus()

# an error is caused and i dont know what causes it. it may be that i got the 'golden feature' wrong and thus it cannot execute properly? anyways the code would
# look like this